In [1]:
library(tidyverse)
library(dbplyr)
library(bigrquery)
library(lubridate)

bq_auth()

project_id = "yhcr-prd-bradfor-bia-core"

# create connection to database
con <- DBI::dbConnect(bigrquery::bigquery(), 
                      project = project_id)

print(paste0("Connected to : ", project_id))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql




[1] "Connected to : yhcr-prd-bradfor-bia-core"


In [2]:
person_ids <- read.csv('data/person_ids.csv', header = TRUE)

## CIN 2009 - 2019

In [3]:
sc_data = "CB_2489.cb_CIN_2009_to_2019"

sc_table <- tbl(con, sc_data) |>
    select(person_id,CIN_ACADYR,CIN_AsylumSeeking,CIN_Disability,CIN_LookedAfterChildAdopted,CIN_CPPindicator,CIN_PrimaryNeedCode) 

In [4]:
sc_df <- collect(sc_table)

In [6]:
head(sc_df)

person_id,CIN_ACADYR,CIN_AsylumSeeking,CIN_Disability,CIN_LookedAfterChildAdopted,CIN_CPPindicator,CIN_PrimaryNeedCode
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
EE16D8E6A3A4653E677A6C1949331DD97821A564C0434E49BC247CDBC622D4DA,2008/2009,2,0,0,0,N1
8ACF32604DD3AE67EEECA75E04FCC614520EFC1D5F4A76CEB0FDF043329DE4D2,2008/2009,2,0,0,0,N1
58E9880BEB3B98F55386269E799E309D2E8E749FCDA17DA02EC68DD0E6120141,2008/2009,2,0,0,0,N1
B84E4C85A13F646A5BF71E39FBBA77F3386205DAAB6AA477B71CEB8F0936E667,2008/2009,2,0,0,0,N5
BB7570010F037F8C712D67E2E7A35608749B2EAF6C32185AAA671947AAB585B4,2008/2009,2,0,0,0,N1
6638113C03496158ABCBAB11BD87929CBF936D378DB60C1211A74E17702899C0,2008/2009,2,0,0,0,N1


In [7]:
sc_df |> arrange(CIN_ACADYR) |> distinct(CIN_ACADYR) |> pull(CIN_ACADYR)

[1] "2008/2009" "2009/2010" "2010/2011" "2011/2012" "2012/2013" "2013/2014"
 [7] "2014/2015" "2015/2016" "2016/2017" "2017/2018" "2018/2019"

In [8]:
sc_df |> distinct(CIN_AsylumSeeking) |> pull(CIN_AsylumSeeking)

[1]  2  1  0 NA

1 = True - the child has been asylum seeking at any time between 1 April 2009 and 31 March 2010

0 = False - the child has not been asylum seeking at any time between 1 April 2009 and 31 March 2010.

2 = ?

Dont use - use field in LAC table instead which covers 2005 onwards (but only for LAC). 

In [10]:
sc_df |> distinct(CIN_Disability) |> pull(CIN_Disability)

[1] 0 1

In [11]:
sc_df |> distinct(CIN_LookedAfterChildAdopted) |> pull(CIN_LookedAfterChildAdopted)

[1]  0  1 NA

In [12]:
sc_df |> distinct(CIN_CPPindicator) |> pull(CIN_CPPindicator)

[1]  0  1 NA

1 = True - if the child is currently the subject of a child protection plan (true at 31 March) 

0 = if the child is currently the subject of a child protection plan (false at 31 March)

In [13]:
sc_df |> distinct(CIN_PrimaryNeedCode) |> pull(CIN_PrimaryNeedCode)

[1] "N1"         "N5"         "N2"         "N9"         "N0"        
 [6] "N4"         "N8"         "N6"         "N3"         "N7"        
[11] NA           "n5"         "N1        " "N8        " "N4        "

N1 = Abuse or neglect
N2 = Child's disability/illness
N3 = Parental Disability/illness
N4 = Family in acute stress
N5 = Family dysfunction
N6 = Socially unacceptable
N7 = Low income
N8 = Absent parenting
N9 = Cases other than Children in Need
N0 = Not stated

In [37]:
sc_df <- sc_df |> 
    mutate(CIN_PrimaryNeedCode = str_trim(CIN_PrimaryNeedCode))

In [39]:
sc_df <- sc_df |> 
    mutate(CIN_PrimaryNeedCode = case_when(
               CIN_PrimaryNeedCode == 'n5' ~ 'N5',
               TRUE ~ CIN_PrimaryNeedCode
            ))

In [40]:
sc_df |> distinct(CIN_PrimaryNeedCode) |> pull(CIN_PrimaryNeedCode)

[1] "N1" "N5" "N2" "N9" "N0" "N4" "N8" "N6" "N3" "N7" NA

In [48]:
sc_filtered <- sc_df |>
    filter(person_id %in% person_ids$person_id) |>
    select(person_id,CIN_ACADYR,CIN_PrimaryNeedCode)

In [49]:
sc_filtered |> group_by(CIN_PrimaryNeedCode) |> tally()

CIN_PrimaryNeedCode,n
<chr>,<int>
N0,408
N1,10562
N2,804
N3,127
N4,849
N5,1240
N6,154
N7,57
N8,109


In [52]:
sc_filtered |> nrow()

[1] 14593

In [50]:
sc_filtered |> 
    summarize(n_distinct(person_id))

n_distinct(person_id)
<int>
4149


In [53]:
sc_merge <- sc_filtered |>
    select(-CIN_ACADYR) |>
    distinct()

In [54]:
sc_merge |> nrow()

[1] 5381

In [55]:
sc_merge |> 
    summarize(n_distinct(person_id))

n_distinct(person_id)
<int>
4149


In [59]:
sc_merge |>
    group_by(person_id) |>
    filter(n() > 1) |>
    arrange(person_id)

person_id,CIN_PrimaryNeedCode
<chr>,<chr>
0019031CD62586D59268772800516ED50CDF1EF7DE46488542BB031850F6E68A,N1
0019031CD62586D59268772800516ED50CDF1EF7DE46488542BB031850F6E68A,N8
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,N1
004E9574803B8B9BCE83304FE77D4DEF97CBE5F974323B03A9626E3F58F2C894,N5
006E9437536191621258B883B68946C2B0D85A0161B4CC2FE397CDE1FC1034FA,N1
006E9437536191621258B883B68946C2B0D85A0161B4CC2FE397CDE1FC1034FA,N9
00748B650176730AAF7158DCC2B2B974C8EB08D2BE8195BBD107F3B670619625,N5
00748B650176730AAF7158DCC2B2B974C8EB08D2BE8195BBD107F3B670619625,N1
00AB4C4DE30808A1DD989DA7B5CE1495F536A534AD3347BB148C1A90D900EE87,N1


In [60]:
ever_cin <- sc_merge |>
    select(person_id) |>
    distinct() |>
    mutate(ever_CIN = TRUE)

In [61]:
ever_cin

person_id,ever_CIN
<chr>,<lgl>
8ACF32604DD3AE67EEECA75E04FCC614520EFC1D5F4A76CEB0FDF043329DE4D2,TRUE
BB7570010F037F8C712D67E2E7A35608749B2EAF6C32185AAA671947AAB585B4,TRUE
5B893168F74F87750BDFC391A40CEBCC88A694306B9CEFD24D7D74DAC8A50788,TRUE
B9FDC8506D505035AFBC556C0DF197EC924E34C886E786F862FE65ADC36D3EBF,TRUE
4CCB3C96D6416EB85EB282210215078F078A1CE0CD06C4FF750A1AC90B53789A,TRUE
4AE5E0FEA4D6E075F07919D3677BBC906824D77A526DAC1F26B52D1B071F6298,TRUE
B39DCE371CCE0D1C88E717FF8EEA9F214138D19C2BB4C0ED0A9D065669A034BA,TRUE
411632455F2940B33496848B3E8BC4D7DB468FEA7D22B0117C3091F405A42A1E,TRUE
C7B88888AF9B4F7C252D0C8CEE12A5D09B8BD8010258B7FBB25774C34E204324,TRUE


In [92]:
lac_extra

person_id,ever_CIN
<chr>,<lgl>
44080A719856D40DB760AFB26616947F2A4045F681170D8445764DB2203C553F,TRUE
00341B29C42EC823E9D06B625E751614D66794F1096E2F045DDE51DB64C9C623,TRUE
3E785E66400648CEBE6958CCCAB9D8674515979299E4185AE32DD24991B5D551,TRUE
1B33F87C130D210228074F35E820081C112EAA5A9068106CE6B6720E6D38CDA1,TRUE
0EB9EE066E0D44B445E65C2A24C10AEEE28CEEA01A3325921EE680B740B85002,TRUE
37C7FFDC4FE97F811B16082A1002151510A18758515AB66FBE79EF7EBDAC1459,TRUE
12EC5AA822D9BD948FEB51E776CEF99595BA202540F6B0E24CADFC28E032AF82,TRUE
0FF1C6B0CA50E3FD91033A1D14A9EFE6B80C9C90F0297A4EB190E58BD35FFC73,TRUE
A7B661ED20298879C2E10E21B03664570242D75694E26CF5CF707799F7EF5880,TRUE


In [93]:
ever_cin <- rbind(ever_cin,lac_extra)

In [95]:
ever_cin |> nrow()

[1] 4159

In [94]:
head(ever_cin)

person_id,ever_CIN
<chr>,<lgl>
8ACF32604DD3AE67EEECA75E04FCC614520EFC1D5F4A76CEB0FDF043329DE4D2,TRUE
BB7570010F037F8C712D67E2E7A35608749B2EAF6C32185AAA671947AAB585B4,TRUE
5B893168F74F87750BDFC391A40CEBCC88A694306B9CEFD24D7D74DAC8A50788,TRUE
B9FDC8506D505035AFBC556C0DF197EC924E34C886E786F862FE65ADC36D3EBF,TRUE
4CCB3C96D6416EB85EB282210215078F078A1CE0CD06C4FF750A1AC90B53789A,TRUE
4AE5E0FEA4D6E075F07919D3677BBC906824D77A526DAC1F26B52D1B071F6298,TRUE


### Save csv

In [96]:
# save as csv
write.csv(ever_cin, "data/ever_cin.csv", row.names = FALSE)

# CIN Disability 2009-2019

In [14]:
dis_data = "CB_2489.cb_CIN_2009_to_2019_Disability"

dis_table <- tbl(con, dis_data) |>
    select(person_id,CIN_ACADYR,CIN_Disability) 

In [27]:
dis_df <- collect(dis_table)

In [28]:
head(dis_df)

person_id,CIN_ACADYR,CIN_Disability
<chr>,<chr>,<chr>
EDA994097F3DDFA87825365C60773712D04CD3654F794D63C55D9E3932D484E0,2008/2009,NA
DBD4209B51C4884D3BCCBA99F753B1B09189BF502241237C59D178D47810FF0A,2008/2009,NA
DAC8EEC48D224CBDDAEA16BB114F1364011C78E6C1B6EF7C8941A3E231F91B16,2008/2009,NA
49C7AB9FEB601D00D97784F5B4B91DAD30B87E65AA9F6777B1FC920EA3AFD43C,2008/2009,NA
54CC7198D9E1CDB2182E47800199B7FA36FADF3695B08BC20DC389F69C703D93,2008/2009,NA
9BBCB3B30E354EE0BBD65262E0DBA7B7654CFA72E3E1643324CE5F60C4FCBDE6,2008/2009,NA


In [24]:
dis_df |> arrange(CIN_ACADYR) |> distinct(CIN_ACADYR) |> pull(CIN_ACADYR)

[1] "2008/2009" "2009/2010" "2010/2011" "2011/2012" "2012/2013" "2013/2014"
 [7] "2014/2015" "2015/2016" "2016/2017" "2017/2018" "2018/2019"

In [25]:
dis_df |> distinct(CIN_Disability) |> pull(CIN_Disability)

[1] NA     "LD"   "PC"   "AUT"  "BEH"  "CON"  "DDA"  "INC"  "MOB"  "VIS" 
[11] "COMM" "HAND" "HEAR" "NONE" "None" "none"

Holds a record of the type of disability(s) a child may suffer from. NONE by itself is used for no disability.

NONE = None
MOB = Mobility
HAND = Hand Function
PC = Personal Care
INC = Incontinence
COMM = Communication
LD = Learning
HEAR = Hearing
VIS = Vision
BEH = Behaviour
CON = Consciousness
AUT = Diagnosed with autism or Aspergers syndrome 
DDA = Disabled under DDA but not in above categories

In [29]:
dis_tidy <- dis_df |> 
    mutate(CIN_Disability = case_when(
        is.na(CIN_Disability) | CIN_Disability %in% c('NONE', 'None', 'none') ~ NA_character_,
        TRUE ~ CIN_Disability
    ))

In [30]:
dis_tidy |> group_by(CIN_Disability) |> tally()

CIN_Disability,n
<chr>,<int>
AUT,1980
BEH,2495
COMM,2878
CON,839
DDA,754
HAND,1064
HEAR,701
INC,1581
LD,4084


In [31]:
dis_filtered <- dis_tidy |>
    filter(person_id %in% person_ids$person_id)

In [32]:
dis_filtered |> group_by(CIN_Disability) |> tally()

CIN_Disability,n
<chr>,<int>
AUT,234
BEH,330
COMM,359
CON,96
DDA,119
HAND,129
HEAR,111
INC,278
LD,436


# SS CIN 

In [33]:
cin_data = "CB_2489.tbl_SocialServices_CiNP"

cin_table <- tbl(con, cin_data) |>
    select(person_id,StartDate,EndDate) 

In [34]:
cin_df <- collect(cin_table)

In [35]:
head(cin_df)

person_id,StartDate,EndDate
<chr>,<chr>,<chr>
5356366A24004B6B14DAAAEE9ADD3EFC661545E444D05C5D4C6001494A3CC372,01/04/2019,
D6B3B45878368C7D67E405D9F608352B0C35D272A9BB8A31DFC350445C58CE91,01/04/2019,27/08/2020
63D51806BF39CD8BA75B167A225A6265CACB64C2BCC450D1CC6A8D1B7762C51A,01/04/2019,28/09/2019
E891B044F3EF5574592575824696FCB0A36088AC566895C92256707227EFE7B4,01/04/2019,30/05/2019
FE31EDBA16EA714C02AF362D2D38E6845D1F61EDA058E3C1FBED9BB816287D4D,01/05/2019,04/07/2019
88B89CC5F7F97D31198DDA1EB490F4726DCD596D645BED3601D35F50530B9B14,01/05/2019,04/07/2019


In [36]:
cin_df |> arrange(StartDate) |> distinct(StartDate) |> pull(StartDate)

[1] "01/02/2021" "01/03/2020" "01/03/2021" "01/04/2019" "01/04/2020"
  [6] "01/04/2021" "01/05/2019" "01/05/2020" "01/06/2020" "01/06/2021"
 [11] "01/07/2019" "01/08/2019" "01/09/2020" "01/10/2019" "01/10/2020"
 [16] "01/11/2019" "01/11/2020" "01/12/2020" "02/01/2020" "02/02/2021"
 [21] "02/03/2020" "02/03/2021" "02/04/2019" "02/04/2020" "02/05/2019"
 [26] "02/05/2021" "02/06/2020" "02/06/2021" "02/07/2019" "02/07/2020"
 [31] "02/08/2019" "02/09/2019" "02/09/2020" "02/10/2019" "02/10/2020"
 [36] "02/11/2020" "02/12/2019" "02/12/2020" "03/01/2020" "03/01/2021"
 [41] "03/02/2020" "03/02/2021" "03/03/2020" "03/03/2021" "03/04/2019"
 [46] "03/04/2020" "03/04/2021" "03/05/2019" "03/05/2021" "03/06/2019"
 [51] "03/06/2020" "03/06/2021" "03/07/2019" "03/07/2020" "03/08/2020"
 [56] "03/09/2019" "03/09/2020" "03/10/2019" "03/11/2019" "03/11/2020"
 [61] "03/12/2019" "03/12/2020" "04/01/2021" "04/02/2020" "04/02/2021"
 [66] "04/03/2020" "04/03/2021" "04/04/2019" "04/05/2020" "04/05/2021"
 [71] "04/06/2019" "04/06/2020" "04/06/2021" "04/07/2019" "04/08/2020"
 [76] "04/09/2019" "04/09/2020" "04/10/2019" "04/11/2019" "04/11/2020"
 [81] "04/12/2019" "04/12/2020" "05/01/2021" "05/02/2020" "05/02/2021"
 [86] "05/03/2020" "05/03/2021" "05/04/2019" "05/05/2020" "05/05/2021"
 [91] "05/06/2019" "05/06/2020" "05/07/2019" "05/08/2019" "05/08/2020"
 [96] "05/09/2019" "05/10/2020" "05/11/2019" "05/11/2020" "05/12/2019"
[101] "06/01/2020" "06/01/2021" "06/02/2020" "06/03/2020" "06/04/2020"
[106] "06/04/2021" "06/05/2019" "06/05/2020" "06/05/2021" "06/06/2019"
[111] "06/07/2020" "06/08/2019" "06/08/2020" "06/09/2019" "06/09/2020"
[116] "06/10/2019" "06/10/2020" "06/11/2019" "06/11/2020" "06/12/2019"
[121] "07/01/2020" "07/01/2021" "07/02/2020" "07/04/2020" "07/04/2021"
[126] "07/05/2019" "07/05/2020" "07/05/2021" "07/06/2021" "07/07/2019"
[131] "07/07/2020" "07/08/2019" "07/09/2020" "07/10/2019" "07/10/2020"
[136] "07/12/2020" "08/01/2020" "08/01/2021" "08/02/2021" "08/03/2021"
[141] "08/04/2019" "08/04/2020" "08/04/2021" "08/05/2019" "08/05/2020"
[146] "08/06/2020" "08/06/2021" "08/07/2019" "08/07/2020" "08/08/2019"
[151] "08/09/2020" "08/10/2019" "08/10/2020" "08/11/2019" "08/11/2020"
[156] "08/12/2020" "09/01/2020" "09/02/2021" "09/03/2020" "09/03/2021"
[161] "09/04/2019" "09/04/2020" "09/04/2021" "09/05/2019" "09/05/2021"
[166] "09/06/2020" "09/06/2021" "09/07/2019" "09/07/2020" "09/08/2019"
[171] "09/09/2019" "09/09/2020" "09/10/2019" "09/10/2020" "09/11/2019"
[176] "09/11/2020" "09/12/2019" "09/12/2020" "10/01/2020" "10/02/2020"
[181] "10/02/2021" "10/03/2020" "10/03/2021" "10/04/2019" "10/05/2019"
[186] "10/05/2021" "10/06/2019" "10/06/2020" "10/07/2019" "10/07/2020"
[191] "10/08/2020" "10/09/2019" "10/09/2020" "10/10/2019" "10/11/2020"
[196] "10/12/2019" "10/12/2020" "11/01/2021" "11/02/2020" "11/02/2021"
[201] "11/03/2020" "11/03/2021" "11/04/2019" "11/04/2021" "11/05/2019"
[206] "11/05/2020" "11/05/2021" "11/06/2019" "11/06/2020" "11/07/2019"
[211] "11/08/2020" "11/09/2019" "11/09/2020" "11/10/2019" "11/11/2019"
[216] "11/11/2020" "11/12/2019" "11/12/2020" "12/01/2021" "12/02/2020"
[221] "12/02/2021" "12/03/2020" "12/03/2021" "12/04/2019" "12/04/2021"
[226] "12/05/2019" "12/05/2020" "12/05/2021" "12/06/2019" "12/07/2019"
[231] "12/08/2020" "12/09/2019" "12/10/2020" "12/11/2019" "12/11/2020"
[236] "12/12/2019" "13/01/2020" "13/01/2021" "13/02/2020" "13/02/2021"
[241] "13/03/2020" "13/04/2019" "13/04/2020" "13/04/2021" "13/05/2019"
[246] "13/05/2020" "13/05/2021" "13/06/2019" "13/07/2019" "13/07/2020"
[251] "13/08/2019" "13/08/2020" "13/09/2019" "13/09/2020" "13/10/2019"
[256] "13/10/2020" "13/11/2019" "13/11/2020" "13/12/2019" "14/01/2020"
[261] "14/01/2021" "14/02/2020" "14/04/2019" "14/04/2020" "14/04/2021"
[266] "14/05/2019" "14/05/2020" "14/05/2021" "14/06/2019" "14/07/2020"
[271] "14/08/2019" "14/08/2020" "14/09/2020" "14/10/2019" "14/10/2020"
[276] "14/11/2019" "14/12/2020" "15/01/2020" "15/01/2021" "15/02/2020"
[281] "15/02/2021" "15/03/

# LAC

In [63]:
lac_data = "CB_2489.cb_CLA_2006_to_2019"

lac_table <- tbl(con, lac_data) |>
    select(person_id,CLA_ACADYR) 

Auto-refreshing stale OAuth token.



In [74]:
lac_df <- collect(lac_table)

In [75]:
head(lac_df)

person_id,CLA_ACADYR
<chr>,<chr>
301A8393CBC9BC5B127B33C9BBC77D89A67932C531AB716D0AC116A29FBC03A7,2005/2006
4C7F526CAEFC48E30751765A0168DFFDF67E717B229E1FFAF0AAADEDA18A3773,2005/2006
243542528EE5BD4E50EEDE9AC7C9587633F5275D27AC69975A43E4634375A484,2005/2006
FF008416E57E330193805A59438F4983E43547DFFF02A162BB31742F5AACEA4D,2005/2006
B93124CA7B49C443CE539682B23536D8F1EB1D35609E1E730AD982CD120D45FB,2005/2006
EE4A1478A07E12766FF1D5DA01DF603A98A4ADE131718D071060F30142D49D3F,2005/2006


In [76]:
lac_df |> arrange(CLA_ACADYR) |> distinct(CLA_ACADYR) |> pull(CLA_ACADYR)

[1] "2005/2006" "2006/2007" "2007/2008" "2008/2009" "2009/2010" "2010/2011"
 [7] "2011/2012" "2012/2013" "2013/2014" "2014/2015" "2015/2016" "2016/2017"
[13] "2017/2018" "2018/2019"

In [78]:
lac_filtered <- lac_df |>
    filter(person_id %in% person_ids$person_id)

In [79]:
ever_lac <- lac_filtered |>
    select(person_id) |>
    distinct() |>
    mutate(ever_LAC = TRUE)

In [80]:
ever_lac |> nrow()

[1] 550

In [82]:
sum(ever_lac$person_id %in% ever_cin$person_id)

[1] 540

In [85]:
lac_extra <- anti_join(ever_lac, ever_cin, by = "person_id")

In [86]:
lac_extra

person_id,ever_LAC
<chr>,<lgl>
44080A719856D40DB760AFB26616947F2A4045F681170D8445764DB2203C553F,TRUE
00341B29C42EC823E9D06B625E751614D66794F1096E2F045DDE51DB64C9C623,TRUE
3E785E66400648CEBE6958CCCAB9D8674515979299E4185AE32DD24991B5D551,TRUE
1B33F87C130D210228074F35E820081C112EAA5A9068106CE6B6720E6D38CDA1,TRUE
0EB9EE066E0D44B445E65C2A24C10AEEE28CEEA01A3325921EE680B740B85002,TRUE
37C7FFDC4FE97F811B16082A1002151510A18758515AB66FBE79EF7EBDAC1459,TRUE
12EC5AA822D9BD948FEB51E776CEF99595BA202540F6B0E24CADFC28E032AF82,TRUE
0FF1C6B0CA50E3FD91033A1D14A9EFE6B80C9C90F0297A4EB190E58BD35FFC73,TRUE
A7B661ED20298879C2E10E21B03664570242D75694E26CF5CF707799F7EF5880,TRUE


In [89]:
lac_filtered |>
    filter(person_id == '3E785E66400648CEBE6958CCCAB9D8674515979299E4185AE32DD24991B5D551')

person_id,CLA_ACADYR
<chr>,<chr>
3E785E66400648CEBE6958CCCAB9D8674515979299E4185AE32DD24991B5D551,2005/2006
3E785E66400648CEBE6958CCCAB9D8674515979299E4185AE32DD24991B5D551,2006/2007


10 extra are dates prior to CIN dataset

In [90]:
lac_extra <- lac_extra |>
    rename(ever_CIN = ever_LAC)